# 01 — Quality | Data-Quality Checks and Reconciliation with Official Standings

**Project:** MLB Performance Intelligence Platform

Runs after Gold. Every check stores its result in `quality.dq_results`; any **CRITICAL** failure
fails the Job (so a broken load is never silently published to the dashboard).

| Group | Checks |
|---|---|
| Completeness | Every completed game has a boxscore; every completed game has exactly two team rows; every player in the facts has a profile |
| Uniqueness | `game_pk` in `fact_game`; `(game_pk, player_id, team_id)` in batting and pitching facts |
| Validity | Completed games have scores and a winner (unless tie); H ≤ AB; HR ≤ H; PA ≥ AB; ER ≤ R |
| Internal reconciliation | Sum of player runs = team runs per game (boxscore vs line score); runs allowed by pitchers = opponent runs |
| **External reconciliation** | Regular-season W, L, runs scored and runs allowed per team computed from games **=** official MLB standings (fully loaded seasons); partly loaded seasons reported as a warning |

In [ ]:
import sys, importlib
_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + _nb.rsplit("/", 2)[0] + "/shared")
import config as C
import pipeline as P
for _m in (C, P):
    importlib.reload(_m)

RUN_ID = P.new_run_id()
T = lambda s, n: C.table(s, n)

def check(check_id, name, layer, table_name, severity, failed_sql, checked_sql, details=""):
    failed = spark.sql(f"SELECT COUNT(*) FROM ({failed_sql})").collect()[0][0]
    checked = spark.sql(checked_sql).collect()[0][0]
    return {"check_id": check_id, "check_name": name, "layer": layer, "table_name": table_name,
            "severity": severity, "failed_rows": failed, "checked_rows": checked, "details": details}

G, TG, BAT, PIT = T("silver", "fact_game"), T("silver", "fact_team_game"), T("silver", "fact_batting_game"), T("silver", "fact_pitching_game")
completed_games = f"SELECT COUNT(*) FROM {G} WHERE is_completed"

In [ ]:
results = [
    check("DQ-001", "Completed game without boxscore", "silver", G, "CRITICAL",
          f"""SELECT g.game_pk FROM {G} g LEFT ANTI JOIN (SELECT DISTINCT game_pk FROM {BAT}) b ON b.game_pk = g.game_pk
              WHERE g.is_completed""", completed_games),
    check("DQ-002", "Duplicate game_pk in fact_game", "silver", G, "CRITICAL",
          f"SELECT game_pk FROM {G} GROUP BY game_pk HAVING COUNT(*) > 1", f"SELECT COUNT(*) FROM {G}"),
    check("DQ-003", "Completed game without two team rows", "silver", TG, "CRITICAL",
          f"SELECT game_pk FROM {TG} WHERE is_completed GROUP BY game_pk HAVING COUNT(*) <> 2", completed_games),
    check("DQ-004", "Completed game missing score or winner", "silver", G, "CRITICAL",
          f"""SELECT game_pk FROM {G} WHERE is_completed
              AND (home_score IS NULL OR away_score IS NULL OR (NOT is_tie AND winning_team_id IS NULL))""", completed_games),
    check("DQ-005", "Duplicate (game_pk, player_id, team_id) in batting", "silver", BAT, "CRITICAL",
          f"SELECT game_pk, player_id, team_id FROM {BAT} GROUP BY ALL HAVING COUNT(*) > 1", f"SELECT COUNT(*) FROM {BAT}"),
    check("DQ-006", "Duplicate (game_pk, player_id, team_id) in pitching", "silver", PIT, "CRITICAL",
          f"SELECT game_pk, player_id, team_id FROM {PIT} GROUP BY ALL HAVING COUNT(*) > 1", f"SELECT COUNT(*) FROM {PIT}"),
    check("DQ-007", "Invalid batting line (H > AB, HR > H or AB > PA)", "silver", BAT, "CRITICAL",
          f"SELECT * FROM {BAT} WHERE h > ab OR hr > h OR ab > pa", f"SELECT COUNT(*) FROM {BAT}"),
    check("DQ-008", "Invalid pitching line (ER > R or negative outs)", "silver", PIT, "CRITICAL",
          f"SELECT * FROM {PIT} WHERE er > r OR outs < 0", f"SELECT COUNT(*) FROM {PIT}"),
    check("DQ-009", "Team runs (line score) differ from sum of player runs (boxscore)", "silver", BAT, "CRITICAL",
          f"""SELECT tg.game_pk, tg.team_id FROM {TG} tg
              JOIN (SELECT game_pk, team_id, SUM(r) AS r FROM {BAT} GROUP BY ALL) b USING (game_pk, team_id)
              WHERE tg.is_completed AND tg.runs_scored <> b.r""", f"SELECT COUNT(*) FROM {TG} WHERE is_completed"),
    check("DQ-010", "Opponent runs differ from runs allowed by pitchers", "silver", PIT, "CRITICAL",
          f"""SELECT tg.game_pk, tg.team_id FROM {TG} tg
              JOIN (SELECT game_pk, team_id, SUM(r) AS r FROM {PIT} GROUP BY ALL) p USING (game_pk, team_id)
              WHERE tg.is_completed AND tg.runs_allowed <> p.r""", f"SELECT COUNT(*) FROM {TG} WHERE is_completed"),
    check("DQ-011", "Player in facts without profile in dim_player", "silver", T("silver", "dim_player"), "WARNING",
          f"""SELECT player_id FROM (SELECT player_id FROM {BAT} UNION SELECT player_id FROM {PIT})
              LEFT ANTI JOIN {T('silver', 'dim_player')} USING (player_id)""",
          f"SELECT COUNT(*) FROM (SELECT player_id FROM {BAT} UNION SELECT player_id FROM {PIT})"),
    check("DQ-012", "Team or venue without dimension row", "silver", T("silver", "dim_team"), "WARNING",
          f"""SELECT team_id FROM (SELECT DISTINCT team_id FROM {TG}) LEFT ANTI JOIN {T('silver', 'dim_team')} USING (team_id)
              UNION ALL
              SELECT venue_id FROM (SELECT DISTINCT venue_id FROM {G}) LEFT ANTI JOIN {T('silver', 'dim_venue')} USING (venue_id)""",
          f"SELECT COUNT(DISTINCT team_id) FROM {TG}"),
]

## External reconciliation: our standings vs official MLB standings
Compared for every team-season whose games are fully loaded (our game count equals the official
`gamesPlayed`). Team-seasons only partly loaded (e.g. a custom date range) are reported separately
as a WARNING instead of a false mismatch.

In [ ]:
recon_sql = f"""
SELECT s.season, s.team_id, s.games_played AS official_g, ts.games AS our_g, s.wins AS official_w, ts.wins AS our_w, s.losses AS official_l, ts.losses AS our_l,
       s.runs_scored AS official_rs, ts.runs_scored AS our_rs, s.runs_allowed AS official_ra, ts.runs_allowed AS our_ra
FROM {T('silver', 'official_standings')} s
JOIN {T('gold', 'team_season_summary')} ts
  ON ts.season = s.season AND ts.team_id = s.team_id AND ts.season_type = 'Regular Season'
"""
spark.sql(recon_sql).createOrReplaceTempView("recon")
results.append(check("DQ-013", "W-L or runs differ from official MLB standings", "gold", T("gold", "team_season_summary"),
    "CRITICAL",
    """SELECT * FROM recon WHERE our_g = official_g
       AND (official_w <> our_w OR official_l <> our_l OR official_rs <> our_rs OR official_ra <> our_ra)""",
    "SELECT COUNT(*) FROM recon WHERE our_g = official_g", "Regular season, per fully loaded team-season"))
results.append(check("DQ-014", "Team-season not fully loaded (fewer games than official standings)", "gold",
    T("gold", "team_season_summary"), "WARNING",
    "SELECT * FROM recon WHERE our_g < official_g", "SELECT COUNT(*) FROM recon",
    "Expected while a season is in progress between runs or when only a date range was loaded"))

In [ ]:
dq = P.record_dq(spark, RUN_ID, results)
display(dq.select("check_id", "check_name", "severity", "status", "failed_rows", "checked_rows").orderBy("check_id"))

critical = [r for r in results if r["severity"] == "CRITICAL" and r["failed_rows"] > 0]
with P.log_step(spark, RUN_ID, "quality", "dq_checks") as step:
    step.records = len(results)
    step.message = f"{len(results)} checks, {len(critical)} critical failure(s)"
    if critical:
        raise RuntimeError("Critical data-quality failures: " + ", ".join(f"{r['check_id']} ({r['failed_rows']})" for r in critical))